In [1]:
import tkinter as tk
import time
import copy
import random

from tkinter import ttk


In [2]:
# Danh sách 24 quận huyện (TP.HCM cũ)
DISTRICTS = [
    'Củ Chi', 'Hóc Môn', 'Quận 12', 'Thủ Đức', 'Quận 9', 'Quận 2', 
    'Bình Thạnh', 'Phú Nhuận', 'Gò Vấp', 'Tân Bình', 'Tân Phú', 
    'Bình Tân', 'Bình Chánh', 'Quận 1', 'Quận 3', 'Quận 4', 'Quận 5', 
    'Quận 6', 'Quận 7', 'Quận 8', 'Quận 10', 'Quận 11', 'Nhà Bè', 'Cần Giờ'
]

# Ràng buộc kề nhau (ước lượng tương đối dựa trên bản đồ TP.HCM)
NEIGHBORS = {
    'Củ Chi': ['Hóc Môn', 'Bình Chánh'],
    'Hóc Môn': ['Củ Chi', 'Bình Chánh', 'Bình Tân', 'Quận 12'],
    'Quận 12': ['Hóc Môn', 'Bình Tân', 'Tân Bình', 'Gò Vấp', 'Bình Thạnh', 'Thủ Đức', 'Tân Phú'],
    'Thủ Đức': ['Quận 12', 'Bình Thạnh', 'Quận 2', 'Quận 9'],
    'Quận 9': ['Thủ Đức', 'Quận 2'],
    'Quận 2': ['Quận 9', 'Thủ Đức', 'Bình Thạnh', 'Quận 1', 'Quận 4', 'Quận 7'],
    'Bình Thạnh': ['Thủ Đức', 'Quận 12', 'Gò Vấp', 'Phú Nhuận', 'Quận 1', 'Quận 2'],
    'Gò Vấp': ['Quận 12', 'Tân Bình', 'Phú Nhuận', 'Bình Thạnh'],
    'Phú Nhuận': ['Gò Vấp', 'Tân Bình', 'Quận 3', 'Quận 1', 'Bình Thạnh'],
    'Tân Bình': ['Quận 12', 'Tân Phú', 'Quận 11', 'Quận 10', 'Quận 3', 'Phú Nhuận', 'Gò Vấp'],
    'Tân Phú': ['Bình Tân', 'Quận 12', 'Tân Bình', 'Quận 11', 'Quận 6'],
    'Bình Tân': ['Bình Chánh', 'Hóc Môn', 'Quận 12', 'Tân Phú', 'Quận 6', 'Quận 8'],
    'Bình Chánh': ['Củ Chi', 'Hóc Môn', 'Bình Tân', 'Quận 8', 'Quận 7', 'Nhà Bè'],
    'Quận 1': ['Bình Thạnh', 'Phú Nhuận', 'Quận 3', 'Quận 5', 'Quận 4', 'Quận 2'],
    'Quận 3': ['Quận 1', 'Phú Nhuận', 'Tân Bình', 'Quận 10'],
    'Quận 10': ['Quận 3', 'Tân Bình', 'Quận 11', 'Quận 5'],
    'Quận 11': ['Tân Phú', 'Tân Bình', 'Quận 10', 'Quận 5', 'Quận 6'],
    'Quận 5': ['Quận 1', 'Quận 10', 'Quận 11', 'Quận 6', 'Quận 8'],
    'Quận 6': ['Bình Tân', 'Tân Phú', 'Quận 11', 'Quận 5', 'Quận 8'],
    'Quận 8': ['Bình Tân', 'Quận 6', 'Quận 5', 'Quận 4', 'Quận 7', 'Bình Chánh'],
    'Quận 4': ['Quận 1', 'Quận 2', 'Quận 7', 'Quận 8'],
    'Quận 7': ['Quận 4', 'Quận 2', 'Quận 8', 'Bình Chánh', 'Nhà Bè'],
    'Nhà Bè': ['Quận 7', 'Bình Chánh', 'Cần Giờ'],
    'Cần Giờ': ['Nhà Bè']
}

# Đảm bảo tính đối xứng của đồ thị (A kề B thì B kề A)
for u in list(NEIGHBORS.keys()):
    for v in NEIGHBORS[u]:
        if v not in NEIGHBORS: NEIGHBORS[v] = []
        if u not in NEIGHBORS[v]: NEIGHBORS[v].append(u)

# Tọa độ tương đối trên Canvas (x, y)
COORDS = {
    'Củ Chi': (50, 50),       'Hóc Môn': (200, 100),     'Quận 12': (400, 100), 
    'Thủ Đức': (750, 70),     'Quận 9': (850, 150),      'Quận 2': (800, 330),
    'Bình Thạnh': (650, 170),  'Gò Vấp': (520, 170),      'Phú Nhuận': (580, 280),
    'Tân Bình': (420, 260),    'Tân Phú': (300, 300),     'Bình Tân': (140, 410),
    'Bình Chánh': (50, 500),   'Quận 1': (680, 370),      'Quận 3': (560, 370),
    'Quận 10': (460, 410),     'Quận 11': (340, 400),     'Quận 5': (450, 510),
    'Quận 6': (290, 500),      'Quận 8': (350, 630),      'Quận 4': (600, 500),
    'Quận 7': (600, 630),      'Nhà Bè': (400, 730),      'Cần Giờ': (650, 750)
}

COLOR_NAMES = ['Đỏ', 'Xanh lá', 'Xanh dương', 'Vàng']
COLOR_CODES = {'Đỏ': '#ff4d4d', 'Xanh lá': '#4dff4d', 'Xanh dương': '#4da6ff', 'Vàng': '#ffff4d', 'Trắng': '#ffffff'}

In [3]:
class CSPSolver:
    def __init__(self, variables, neighbors, domains_list, log_callback, paint_callback):
        self.variables = variables
        self.neighbors = neighbors
        self.domains_list = domains_list
        self.log_cb = log_callback
        self.paint_cb = paint_callback
        self.step_counter = 1

    def format_assignment(self, assignment):
        return "{" + ", ".join([f"{k}={v}" for k, v in assignment.items()]) + "}"

    def format_brief(self, current):
        items = list(current.items())[:6]
        text = ", ".join([f"{k}={v}" for k, v in items])
        return text + ", ..." if len(current) > 6 else text

    # --- THUẬT TOÁN 1: BACKTRACKING ---
    def solve_backtracking(self, assignment, domains):
        if len(assignment) == len(self.variables):
            return True

        unassigned_vars = [v for v in self.variables if v not in assignment]
        var = unassigned_vars[0] 
        
        self.log_cb(f"\nBước {self.step_counter}: Chọn biến {var}")
        self.step_counter += 1

        for color in domains[var]:
            self.log_cb(f"  + Thử gán: {var} = {color}")
            
            is_valid = True
            violated_neighbor = None
            
            for neighbor in self.neighbors[var]:
                if neighbor in assignment and assignment[neighbor] == color:
                    is_valid = False
                    violated_neighbor = neighbor
                    break
            
            if not is_valid:
                self.log_cb(f"  -> Không hợp lệ (Vi phạm ràng buộc: {var} ≠ {violated_neighbor} đang cùng màu {color})")
                continue 
                
            self.log_cb(f"  -> Hợp lệ")
            assignment[var] = color
            self.paint_cb(var, color)
            self.log_cb(f"  -> Assignment = {self.format_assignment(assignment)}")

            result = self.solve_backtracking(assignment, domains)
            if result:
                return True

            del assignment[var]
            self.paint_cb(var, 'Trắng')
            self.log_cb(f"\nBước {self.step_counter}: QUAY LUI (Backtrack)!")
            self.log_cb(f"  + Hủy gán {var}={color}, loại bỏ hướng đi này.")
            self.step_counter += 1

        return False

    # --- THUẬT TOÁN 2: BACKTRACKING + FORWARD CHECKING ---
    def solve_forward_checking(self, assignment, domains):
        if len(assignment) == len(self.variables):
            return True

        unassigned_vars = [v for v in self.variables if v not in assignment]
        var = unassigned_vars[0] 
        
        self.log_cb(f"\nBước {self.step_counter}: Chọn biến {var}")
        self.step_counter += 1

        for color in domains[var]:
            self.log_cb(f"  + Thử gán: {var} = {color}")
            
            is_valid = True
            violated_neighbor = None
            
            for neighbor in self.neighbors[var]:
                if neighbor in assignment and assignment[neighbor] == color:
                    is_valid = False
                    violated_neighbor = neighbor
                    break
            
            if not is_valid:
                self.log_cb(f"  -> Không hợp lệ (Vi phạm ràng buộc: {var} ≠ {violated_neighbor} đang cùng màu {color})")
                continue 
                
            self.log_cb(f"  -> Hợp lệ")
            assignment[var] = color
            self.paint_cb(var, color)
            self.log_cb(f"  -> Assignment = {self.format_assignment(assignment)}")

            new_domains = copy.deepcopy(domains)
            new_domains[var] = [color]
            
            fc_success = True
            updated_domains = []
            empty_domain_var = None

            for neighbor in self.neighbors[var]:
                if neighbor not in assignment:
                    if color in new_domains[neighbor]:
                        new_domains[neighbor].remove(color)
                        updated_domains.append(neighbor)
                        if len(new_domains[neighbor]) == 0:
                            fc_success = False
                            empty_domain_var = neighbor

            if updated_domains:
                self.log_cb("  -> Update domain các biến kề:")
                for neighbor in updated_domains:
                    domain_str = "{" + ", ".join(new_domains[neighbor]) + "}"
                    self.log_cb(f"     * Miền giá trị của {neighbor} = {domain_str}")

            if fc_success:
                result = self.solve_forward_checking(assignment, new_domains)
                if result:
                    return True
            else:
                self.log_cb(f"  => CẢNH BÁO: Miền giá trị của {empty_domain_var} bị rỗng (Forward Checking)!")

            del assignment[var]
            self.paint_cb(var, 'Trắng')
            self.log_cb(f"\nBước {self.step_counter}: QUAY LUI (Backtrack)!")
            self.log_cb(f"  + Hủy gán {var}={color}, loại bỏ hướng đi này.")
            self.step_counter += 1

        return False

    # --- THUẬT TOÁN 3: AC-3 (MAC - Maintaining Arc Consistency) ---
    def ac3(self, domains, queue):
        self.log_cb("\n  => Chạy AC-3:")
        self.log_cb("     B1: Khởi tạo 1 hàng đợi Q chứa các cung cần xét.")
        
        while queue:
            (xi, xj) = queue.pop(0)
            self.log_cb(f"     B2: Kiểm tra cung ({xi}, {xj})")
            
            revised = False
            for x in domains[xi][:]:
                if len(domains[xj]) == 1 and domains[xj][0] == x:
                    domains[xi].remove(x)
                    revised = True
                    
            if revised:
                self.log_cb(f"       -> Miền giá trị của {xi} bị thay đổi thành {domains[xi]}")
                if len(domains[xi]) == 0:
                    self.log_cb(f"       -> Domain của {xi} = {{}} => THẤT BẠI")
                    return False
                self.log_cb(f"       -> Thêm vào Q tất cả các cung kết thúc bằng {xi}")
                for xk in self.neighbors[xi]:
                    if xk != xj:
                        queue.append((xk, xi))
                        
        self.log_cb("     B3: Hàng đợi Q = {} => THÀNH CÔNG (Không vi phạm)")
        return True

    def solve_ac3_mac(self, assignment, domains):
        if len(assignment) == len(self.variables): return True

        unassigned_vars = [v for v in self.variables if v not in assignment]
        var = unassigned_vars[0] 
        self.log_cb(f"\nBước {self.step_counter}: Chọn biến {var}"); self.step_counter += 1

        for color in domains[var]:
            self.log_cb(f"  + Thử gán: {var} = {color}")
            is_valid = True
            for neighbor in self.neighbors[var]:
                if neighbor in assignment and assignment[neighbor] == color:
                    is_valid = False; break
            
            if not is_valid: continue 
                
            assignment[var] = color
            self.paint_cb(var, color)

            new_domains = copy.deepcopy(domains)
            new_domains[var] = [color]
            
            queue = [(neighbor, var) for neighbor in self.neighbors[var] if neighbor not in assignment]
            
            if self.ac3(new_domains, queue):
                if self.solve_ac3_mac(assignment, new_domains): return True

            del assignment[var]
            self.paint_cb(var, 'Trắng')
            self.log_cb(f"  + QUAY LUI: Hủy gán {var}={color}")

        return False

    # --- THUẬT TOÁN 4: MIN-CONFLICTS (Local Search) ---
    def solve_min_conflicts(self, max_steps=1000):
        current = {var: random.choice(self.domains_list) for var in self.variables}
        self.log_cb("Bước 1. Khởi tạo lời giải ban đầu (ngẫu nhiên)")
        self.log_cb(f"Giả sử ta gán ban đầu: {self.format_brief(current)}")
        
        for node, color in current.items():
            self.paint_cb(node, color, delay=0.01) 
            
        for step in range(1, max_steps + 1):
            conflicts = []
            violated_edges = set()
            
            self.log_cb(f"\nKiểm tra ràng buộc (Lần lặp {step}):")
            
            log_limit = 5 
            printed = 0
            
            for u in self.variables:
                for v in self.neighbors[u]:
                    if u < v: 
                        if current[u] == current[v]:
                            conflicts.extend([u, v])
                            violated_edges.add((u, v))
                            if printed < log_limit:
                                self.log_cb(f"{u} ≠ {v} -> vi phạm ({current[u]} = {current[v]})")
                                printed += 1
                        else:
                            if printed < log_limit:
                                self.log_cb(f"{u} ≠ {v} -> thỏa mãn")
                                printed += 1
            
            conflicted_vars = list(set(conflicts))
            
            if not conflicted_vars:
                self.log_cb("\n=> KHÔNG CÒN XUNG ĐỘT. THÀNH CÔNG!")
                return True
                
            self.log_cb(f"... Như vậy, có tổng cộng {len(violated_edges)} xung đột.")
            
            var = random.choice(conflicted_vars)
            self.log_cb("\nBước 2. Chọn một biến gây xung đột")
            self.log_cb(f"Chọn ngẫu nhiên một biến trong cặp xung đột. Giả sử chọn {var}.")
            
            self.log_cb(f"Bước 3. Tìm giá trị làm giảm xung đột bằng cách thử tất cả các giá trị trong miền của {var}:")
            
            min_c = float('inf')
            best_colors = []
            
            for color in self.domains_list:
                c = sum(1 for neighbor in self.neighbors[var] if current[neighbor] == color)
                if c > 0:
                    self.log_cb(f"{var}={color} -> xung đột với {c} biến kề")
                else:
                    self.log_cb(f"{var}={color} -> thỏa mãn (0 xung đột)")
                    
                if c < min_c:
                    min_c = c
                    best_colors = [color]
                elif c == min_c:
                    best_colors.append(color)
            
            best_color = random.choice(best_colors)
            self.log_cb(f"Như vậy, chọn giá trị tốt nhất để gán cho biến {var} là {best_color}.")
            
            self.log_cb("Bước 4. Cập nhật lời giải")
            current[var] = best_color
            self.paint_cb(var, best_color, delay=0.3)
            self.log_cb(f"Gán lại: {var}={best_color}")
            self.log_cb("LẶP LẠI Kiểm tra lại ràng buộc")

        self.log_cb("\n=> ĐẠT GIỚI HẠN SỐ BƯỚC (MAX_STEPS). THẤT BẠI!")
        return False

In [4]:
class MapColoringApp:
    def __init__(self, root):
        self.root = root
        self.root.title("CSP - Map Coloring (Đã tách thuật toán & Thêm Min-Conflicts/AC3)")
        self.root.geometry("1400x900")

        self.setup_ui()
        self.node_shapes = {}
        self.draw_map()

    def setup_ui(self):
        self.left_frame = tk.Frame(self.root, bg='#f0f0f0')
        self.left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        self.right_frame = tk.Frame(self.root, width=500, bg='white')
        self.right_frame.pack(side=tk.RIGHT, fill=tk.Y)
        self.right_frame.pack_propagate(False)

        self.canvas = tk.Canvas(self.left_frame, bg='#f0f0f0')
        self.canvas.pack(fill=tk.BOTH, expand=True)

        self.algo_var = tk.StringVar(value="Backtracking")
        self.combo = ttk.Combobox(self.left_frame, textvariable=self.algo_var, 
                                  values=["Backtracking", 
                                          "Backtracking + ForwardChecking",
                                          "Backtracking + AC-3",
                                          "Min-Conflicts"], 
                                  state="readonly", font=("Arial", 12))
        self.combo.pack(pady=5)

        self.btn_start = tk.Button(self.left_frame, text="Bắt đầu tô màu", command=self.start_csp, 
                                   font=("Arial", 12, "bold"), bg="#4CAF50", fg="white")
        self.btn_start.pack(pady=10)

        tk.Label(self.right_frame, text="Log thực thi thuật toán", font=("Arial", 14, "bold"), bg='white').pack(pady=5)
        self.log_text = tk.Text(self.right_frame, font=("Consolas", 10), wrap=tk.WORD, padx=5, pady=5)
        self.scrollbar = tk.Scrollbar(self.right_frame, command=self.log_text.yview)
        self.log_text.configure(yscrollcommand=self.scrollbar.set)
        
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

    def draw_map(self):
        drawn_edges = set()
        for u in DISTRICTS:
            for v in NEIGHBORS[u]:
                edge = tuple(sorted((u, v)))
                if edge not in drawn_edges:
                    x1, y1 = COORDS[u]
                    x2, y2 = COORDS[v]
                    self.canvas.create_line(x1, y1, x2, y2, fill="gray", dash=(4, 4))
                    drawn_edges.add(edge)

        for node in DISTRICTS:
            x, y = COORDS[node]
            w, h = 32, 16
            
            self.canvas.create_rectangle(x-w+3, y-h+3, x+w+3, y+h+3, fill='#d9d9d9', outline='')
            rect = self.canvas.create_rectangle(x-w, y-h, x+w, y+h, fill='white', outline='black', width=1.5)
            self.canvas.create_text(x, y, text=node, font=("Arial", 8, "bold"))
            self.node_shapes[node] = rect

    def write_log(self, text):
        self.log_text.insert(tk.END, text + "\n")
        self.log_text.see(tk.END)
        self.root.update()

    def update_node_color(self, node, color_name, delay=0.3):
        hex_color = COLOR_CODES.get(color_name, "white")
        self.canvas.itemconfig(self.node_shapes[node], fill=hex_color)
        self.root.update()
        time.sleep(delay) 

    def start_csp(self):
        self.btn_start.config(state=tk.DISABLED)
        self.combo.config(state=tk.DISABLED)
        self.log_text.delete(1.0, tk.END)
        
        for node in DISTRICTS:
            self.update_node_color(node, 'Trắng', delay=0)

        algo_choice = self.algo_var.get()
        self.write_log(f"--- Đang chạy thuật toán: {algo_choice} ---\n")

        solver = CSPSolver(
            variables=DISTRICTS,
            neighbors=NEIGHBORS,
            domains_list=COLOR_NAMES,
            log_callback=self.write_log,
            paint_callback=self.update_node_color
        )

        assignment = {}
        domains = {node: list(COLOR_NAMES) for node in DISTRICTS}
        
        if algo_choice == "Backtracking":
            self.write_log(f"Bước {solver.step_counter}: Bắt đầu phép gán rỗng. Assignment={{}}")
            solver.step_counter += 1
            success = solver.solve_backtracking(assignment, domains)
        elif algo_choice == "Backtracking + ForwardChecking":
            self.write_log(f"Bước {solver.step_counter}: Bắt đầu phép gán rỗng. Assignment={{}}")
            solver.step_counter += 1
            success = solver.solve_forward_checking(assignment, domains)
        elif algo_choice == "Backtracking + AC-3":
            self.write_log(f"Bước {solver.step_counter}: Bắt đầu phép gán rỗng. Assignment={{}}")
            solver.step_counter += 1
            success = solver.solve_ac3_mac(assignment, domains)
        elif algo_choice == "Min-Conflicts":
            success = solver.solve_min_conflicts()
        
        if success:
            self.write_log("\n=> HOÀN THÀNH TÔ MÀU BẢN ĐỒ TẤT CẢ RÀNG BUỘC!")
        else:
            self.write_log("\n=> KHÔNG TÌM THẤY GIẢI PHÁP HOẶC THẤT BẠI!")
            
        self.btn_start.config(state=tk.NORMAL)
        self.combo.config(state="readonly")

In [5]:
# Khởi chạy ứng dụng
if __name__ == "__main__":
    root = tk.Tk()
    app = MapColoringApp(root)
    root.mainloop()